# Data Preparation

## Purpose

The purpose of this notebook is to prepare the selected Fitbit datasets for analysis.

Based on the findings documented in the Data Inventory notebook, the two Fitbit exports will be merged into a single master dataset. The preparation phase will focus on validating record integrity, identifying duplicate observations, checking for missing values, and creating clean datasets suitable for analysis.

The outputs generated in this notebook will be stored in the data/processed directory and will serve as the foundation for all subsequent analyses.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

## Load Daily Activity Datasets

The daily activity dataset was selected as the primary dataset for analysis because it contains the most comprehensive collection of user activity metrics, including steps, calories, distance, and activity intensity measures.

The two Fitbit exports will be loaded and evaluated before consolidation.

In [3]:
export1 = Path(r"C:\Users\Weekseey\Documents\Data Projects\bellabeat-analysis\data\raw\fitbit_export_3.12.16-4.11.16")

export2 = Path(r"C:\Users\Weekseey\Documents\Data Projects\bellabeat-analysis\data\raw\fitbit_export_4.12.16-5.12.16")

daily1 = pd.read_csv(export1 / "dailyActivity_merged.csv")
daily2 = pd.read_csv(export2 / "dailyActivity_merged.csv")

In [4]:
print("Export 1 Shape:", daily1.shape)
print("Export 2 Shape:", daily2.shape)

Export 1 Shape: (457, 15)
Export 2 Shape: (940, 15)


## Validate Dataset Compatibility

Before merging the datasets, it is important to verify that both exports share the same structure and column definitions.

Compatible schemas ensure that records can be combined without introducing inconsistencies or data loss.

In [5]:
daily1.columns.equals(daily2.columns)

True

In [6]:
daily1.dtypes.equals(daily2.dtypes)

True

## Preliminary Validation Results

The two daily activity datasets share identical column structures and data types.

Because the datasets represent consecutive collection periods and maintain schema consistency, they are suitable candidates for consolidation into a single master dataset.

## Duplicate Record Check

Before merging the two daily activity datasets, I will check for duplicate records based on user ID and activity date.

Each user should have no more than one daily activity record per date. Duplicate user-date combinations could inflate activity totals or distort averages.

In [7]:
duplicate_keys_1 = daily1.duplicated(subset=["Id", "ActivityDate"]).sum()
duplicate_keys_2 = daily2.duplicated(subset=["Id", "ActivityDate"]).sum()

print("Duplicate user-date records in Export 1:", duplicate_keys_1)
print("Duplicate user-date records in Export 2:", duplicate_keys_2)

Duplicate user-date records in Export 1: 0
Duplicate user-date records in Export 2: 0


In [8]:
daily1_keys = set(zip(daily1["Id"], daily1["ActivityDate"]))
daily2_keys = set(zip(daily2["Id"], daily2["ActivityDate"]))

overlapping_records = daily1_keys.intersection(daily2_keys)

print("Overlapping user-date records between exports:", len(overlapping_records))

Overlapping user-date records between exports: 24


In [9]:
overlap_df1 = daily1[daily1.set_index(["Id", "ActivityDate"]).index.isin(overlapping_records)]
overlap_df2 = daily2[daily2.set_index(["Id", "ActivityDate"]).index.isin(overlapping_records)]

overlap_df1_sorted = overlap_df1.sort_values(["Id", "ActivityDate"]).reset_index(drop=True)
overlap_df2_sorted = overlap_df2.sort_values(["Id", "ActivityDate"]).reset_index(drop=True)

overlap_df1_sorted.equals(overlap_df2_sorted)

False

In [11]:
comparison = overlap_df1_sorted.compare(overlap_df2_sorted)

comparison

TotalSteps        TotalDistance        TrackerDistance         \
         self  other          self  other            self  other   
0         224  13162          0.14   8.50            0.14   8.50   
1        6627   8163          4.31   5.31            4.31   5.31   
2           0   6697          0.00   4.43            0.00   4.43   
3          24    678          0.02   0.47            0.02   0.47   
4        6717  11875          4.72   8.34            4.72   8.34   
5        1019   4414          0.63   2.74            0.63   2.74   
6        2098  10725          1.41   7.49            1.41   7.49   
7           0  10113          0.00   6.83            0.00   6.83   
8           0   8796          0.00   5.91            0.00   5.91   
9         759   8856          0.57   5.98            0.57   5.98   
10          8   8539          0.01   6.12            0.01   6.12   
11        187   5394          0.14   4.03            0.14   4.03   
12        278   3276          0.19   2.20            0.19   2.20   
13       1260   5135          0.83   3.39            0.83   3.39   
14          0   7213          0.00   5.88            0.00   5.88   
15       3436  11596          2.24   7.57            2.24   7.57   
16       5893  10199          3.90   6.74            3.90   6.74   
17       7413  14172          5.77  10.29            4.96   9.48   
18        430  11317          0.26   8.41            0.26   8.41   
19        290  18060          0.21  14.12            0.21  14.12   
20          0   9033          0.00   7.16            0.00   7.16   
21       3246   7626          2.57   6.05            2.57   6.05   
22         20   2564          0.01   1.64            0.01   1.64   
23       2350  23186          1.78  20.40            1.78  20.40   

   VeryActiveDistance        ModeratelyActiveDistance        ...  \
                 self  other                     self other  ...   
0                0.00   1.88                     0.00  0.55  ...   
1                 NaN    NaN                      NaN   NaN  ...   
2                 NaN    NaN                      NaN   NaN  ...   
3                 NaN    NaN                      NaN   NaN  ...   
4                3.23   3.31                     0.22  0.77  ...   
5                0.00   0.19                     0.00  0.35  ...   
6                0.00   1.17                     0.00  0.31  ...   
7                0.00   2.00                     0.00  0.62  ...   
8                0.00   0.11                     0.00  0.93  ...   
9                0.00   3.06                     0.00  0.91  ...   
10               0.00   0.15                     0.00  0.24  ...   
11                NaN    NaN                      NaN   NaN  ...   
12                NaN    NaN                      NaN   NaN  ...   
13                NaN    NaN                      NaN   NaN  ...   
14                NaN    NaN                      NaN   NaN  ...   
15               0.50   1.37                     0.67  0.79  ...   
16               2.88   3.40                     0.56  0.83  ...   
17                NaN    NaN                     0.34  0.38  ...   
18               0.00   5.27                     0.00  0.15  ...   
19               0.00  11.64                     0.00  0.39  ...   
20               0.00   5.43                     0.00  0.14  ...   
21                NaN    NaN                      NaN   NaN  ...   
22                NaN    NaN                      NaN   NaN  ...   
23               0.00  12.22                     0.00  0.34  ...   

   VeryActiveMinutes        FairlyActiveMinutes       LightlyActiveMinutes  \
                self  other                self other                 self   
0                0.0   25.0                 0.0  13.0                    9   
1                NaN    NaN                 NaN   NaN                   89   
2                NaN    NaN                 NaN   NaN                    0   
3                NaN    NaN                 NaN   NaN                    3 

## Overlapping Record Decision

Although no duplicate user-date records were found within each individual export, 24 overlapping user-date records were identified between Export 1 and Export 2.

A comparison of these overlapping records showed that the values were not identical. Export 2 generally contained more complete activity totals for the overlapping user-date records.

### Decision

For overlapping user-date records, the Export 2 version will be retained and the Export 1 version will be removed before consolidation.

### Rationale

Export 2 appears to contain more complete daily activity records for overlapping dates. Retaining Export 2 values reduces the risk of undercounting user activity.

## Dataset Consolidation

Based on the data inventory findings, the two daily activity exports will be consolidated into a single master dataset.

To avoid duplicate observations:

- Non-overlapping records from Export 1 will be retained.
- Overlapping user-date records from Export 1 will be removed.
- Export 2 records will be retained in full.

This approach preserves the most complete activity information available while maximizing the observation period.

In [12]:
daily1_filtered = daily1[
    ~daily1.set_index(["Id", "ActivityDate"]).index.isin(overlapping_records)
]

print("Export 1 Original Rows:", len(daily1))
print("Export 1 Rows After Removing Overlaps:", len(daily1_filtered))

Export 1 Original Rows: 457
Export 1 Rows After Removing Overlaps: 433


In [13]:
master_daily_activity = pd.concat(
    [daily1_filtered, daily2],
    ignore_index=True
)

print("Master Dataset Rows:", len(master_daily_activity))
print("Master Dataset Columns:", len(master_daily_activity.columns))

Master Dataset Rows: 1373
Master Dataset Columns: 15


In [14]:
duplicates = master_daily_activity.duplicated(
    subset=["Id", "ActivityDate"]
).sum()

print("Duplicate User-Date Records:", duplicates)

Duplicate User-Date Records: 0


In [15]:
print(
    "Unique Users:",
    master_daily_activity["Id"].nunique()
)

Unique Users: 35


In [16]:
print(
    "Start Date:",
    master_daily_activity["ActivityDate"].min()
)

print(
    "End Date:",
    master_daily_activity["ActivityDate"].max()
)

Start Date: 3/12/2016
End Date: 5/9/2016


## Consolidation Results

The two Fitbit daily activity exports were successfully consolidated into a single master dataset.

### Final Dataset Summary

- Total Records: 1,373
- Unique Users: 35
- Duplicate User-Date Records: 0
- Observation Period: March 12, 2016 through May 9, 2016

The resulting master dataset will be used as the primary source for activity-related analyses throughout the remainder of the project.

In [17]:
processed_path = Path(
    r"C:\Users\Weekseey\Documents\Data Projects\bellabeat-analysis\data\processed"
)

processed_path.mkdir(parents=True, exist_ok=True)

master_daily_activity.to_csv(
    processed_path / "master_daily_activity.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.
